# Perform a simple Gaia Query

In [ ]:
from astroquery.gaia import Gaia
import astropy.units as u
from astropy.coordinates import SkyCoord
Gaia.MAIN_GAIA_TABLE = "gaiadr3.gaia_source"
Gaia.ROW_LIMIT = -1

def getGaiaDR3data(input_ra, input_dec, radius_search, gaia_params=None, input_ra_unit=u.hour, check_gaia_params=False):
    """
    Simple query function to retrieve Gaia data.
    note: I did this before start to using the SQL query
    
    Parameters:
    input_ra: RA of the centre of the field to retrieve, in hh:mm:ss or deg format
    input_dec: Dec of the centre of the field to retrieve, in deg format
    radius_search: radius of the area to search, in deg
    gaia_params: list with the parameters to retrieve from gaia. If None, a minimal list of parameters is created.
    input_ra_unit: astropy units of the input ra, default u.hour
    check_gaia_params: if True print the name of all Gaia columns retrieved. Default False.
    
    Returns:
    output: a dictionary with the result of the query. Each entry of the dictionary corresponds to one of the params
            specified in gaia_params
    """
    
    coord = SkyCoord(ra=input_ra, dec=input_dec, unit=(input_ra_unit, u.degree), frame='icrs')
    j = Gaia.cone_search_async(coord, radius=u.Quantity(radius_search, u.degree))
    r = j.get_results()
    
    if check_gaia_params:
        for column in r.columns:
            print(column)
    
    if gaia_params is None:
        gaia_params = ['SOURCE_ID', 'ra', 'dec', 'phot_g_mean_mag']  # minimal Gaia parameter list
    
    output = {}
    for param in gaia_params:
        output[param] = r[param]
        
    return output

def getGaiaDR3andGSPCdata(input_ra, input_dec, radius_search, mag_band, mag_max, query_str=None):
    """
    Simple query function to retrieve Gaia data.
    It performs a Gaia query over a region in the sky and retrieve basic data from the Gaia Data Release 3 (DR3) 
    and the Gaia Synthetic Photometry Catalogue (GSPC).
    From the latter it returns the sdss magnitudes.
    
    note: this method works better than the conesearch I used before.
    
    Parameters:
    input_ra: RA of the centre of the field to retrieve, in deg format
    input_dec: Dec of the centre of the field to retrieve, in deg format
    radius_search: radius of the area to search, in deg
    query_str: string passed to the query, If None a default query is used.
    
    Returns:
    output: the resulting table from the Gaia query.
    """
    
    query_conesearch = str(input_ra)+','+str(input_dec)+','+str(radius_search)
    
    if query_str is None:
        query_str="""       
SELECT dr3.source_id, dr3.ra, dr3.dec, classprob_dsc_combmod_star,
gspc.source_id, gspc.u_sdss_mag, gspc.g_sdss_mag, gspc.r_sdss_mag, gspc.i_sdss_mag
FROM gaiadr3.gaia_source AS dr3
JOIN gaiadr3.synthetic_photometry_gspc AS gspc
ON gspc.source_id = dr3.source_id
WHERE 1=CONTAINS(
  POINT('ICRS', dr3.ra, dr3.dec),
  CIRCLE('ICRS', {})
)  
AND
    {} < {}
        """.format(query_conesearch, str(mag_band), str(mag_max))
    
    print('Query used:\n')
    print(query_str)
    job = Gaia.launch_job_async(query_str)
    return job.get_results()


In [ ]:
def getGaiaDR3andGSPCdataBox(input_ra, input_dec, r_ra, r_dec, bands_list=[], mag_band='g_sdss_mag', mag_max=19.5, query_str=None, check_gaia_params=False):
    """
    Simple query function to retrieve Gaia data.
    It performs a Gaia query over a region in the sky and retrieve basic data from the Gaia Data Release 3 (DR3) 
    and the Gaia Synthetic Photometry Catalogue (GSPC).
    From the latter it returns the sdss magnitudes.
    
    note: this method works better than the conesearch I used before.
    
    Parameters:
    input_ra: RA of the centre of the field to retrieve, in deg format
    input_dec: Dec of the centre of the field to retrieve, in deg format
    r_ra: half RA size of the area to search, in deg
    r_dec: half DEC size of the area to search, in deg
    query_str: string passed to the query, If None a default query is used.
    mag_band: string giving the name of the band to limit query ['g_sdss_mag']
    mag_max: magnitude of faintest star to be retrieved [float: 19.5]
    
    
    Returns:
    output: the resulting table from the Gaia query [astropy.table.table.Table]
    """
    
    #query_conesearch = str(input_ra)+','+str(input_dec)+','+str(radius_search)
    r1_str = str(input_ra - r_ra)
    r2_str = str(input_ra + r_ra)
    dec1_str = str(input_dec - r_dec)
    dec2_str = str(input_dec + r_dec)

    # we create a string with the requested band fields to pass to query
    bands_string = ', '.join(bands_list)
    if mag_band not in bands_string:
        print('conditioning band {mag_band} not in list of bands requested.')
        return None
    print(bands_string)
    
    if query_str is None:
        query_str2=f"""       
SELECT dr3.source_id, dr3.ra, dr3.dec, classprob_dsc_combmod_star,
gspc.source_id, {bands_string}
FROM gaiadr3.gaia_source AS dr3
JOIN gaiadr3.synthetic_photometry_gspc AS gspc
ON gspc.source_id = dr3.source_id
WHERE 
    ra BETWEEN {r1_str} AND {r2_str}
AND
    dec BETWEEN {dec1_str} AND {dec2_str}
AND
    {mag_band} < {str(mag_max)}
        """
    
    print('Query used:\n')
    print(query_str2)
    job = Gaia.launch_job_async(query_str2)
    results = job.get_results()
    if check_gaia_params:
        for column in results.columns:
            print(column)
            
    return results

In [ ]:
# Just for test, do a query on a random place.
# plot the name of all columns retrieved 

ra = 200.0
dec = -45.0
r = 0.1  # degree
required_params = ['SOURCE_ID', 'ra', 'dec', # basic parameters
                   'phot_g_mean_mag', 'phot_bp_mean_mag', 'phot_rp_mean_mag', # mag parameters
                   'classprob_dsc_combmod_star'  # star probability classification
                  ]
gaiatest = getGaiaDR3data(input_ra=ra, input_dec=dec, radius_search=r, 
                       gaia_params=required_params, input_ra_unit=u.degree, check_gaia_params=True)


In [ ]:
# and then check the dictionary with the params
for param in gaiatest.keys():
    print(gaiatest[param])  # data in the tables can be accessed with gaiatest[param].data

In [ ]:
# now lets test the second function
# testing over the CP Pup field
#ra = '08 11 46.06'
#dec = '-35 21 04.98'
# the ra/dec must be in degrees so I will transform it using the SkyCoord class
#c = SkyCoord(ra, dec, frame='icrs', unit=(u.hourangle, u.deg))
#ra_deg = c.ra.deg
#dec_deg = c.dec.deg

ra_deg = 132.8522154348
dec_deg = 11.83054623453
c = SkyCoord(ra_deg, dec_deg, frame='icrs', unit=(u.deg, u.deg))
ra_deg = c.ra.deg
dec_deg = c.dec.deg


r = 0.8  # degree

bands_list_sdss = ["gspc.u_sdss_mag", "gspc.u_sdss_flux", "gspc.u_sdss_flux_error",
                   "gspc.g_sdss_mag", "gspc.g_sdss_flux", "gspc.g_sdss_flux_error",
                   "gspc.r_sdss_mag", "gspc.r_sdss_flux", "gspc.r_sdss_flux_error",
                   "gspc.i_sdss_mag", "gspc.i_sdss_flux", "gspc.i_sdss_flux_error",
                   "gspc.z_sdss_mag", "gspc.z_sdss_flux", "gspc.z_sdss_flux_error"]

bands_list_jkc = ["gspc.u_jkc_mag", "gspc.u_jkc_flux", "gspc.u_jkc_flux_error",
                  "gspc.b_jkc_mag", "gspc.b_jkc_flux", "gspc.b_jkc_flux_error",
                  "gspc.v_jkc_mag", "gspc.v_jkc_flux", "gspc.v_jkc_flux_error",
                  "gspc.r_jkc_mag", "gspc.r_jkc_flux", "gspc.r_jkc_flux_error",
                  "gspc.i_jkc_mag", "gspc.i_jkc_flux", "gspc.i_jkc_flux_error"]

bands_string = ', '.join(bands_list)
print(bands_string)

gaiatest = getGaiaDR3andGSPCdataBox(input_ra=ra_deg, input_dec=dec_deg, r_ra=r, r_dec=r, bands_list=bands_list_jkc, mag_band='u_jkc_mag', mag_max=19.5,check_gaia_params=True)
gaiatest

In [ ]:
# lets create a couple of diagram to check the data
import matplotlib.pyplot as plt
import numpy as np

print(type(gaiatest))
ra = gaiatest['ra'].data
de = gaiatest['dec'].data
print(len(ra))
r_centre = np.sqrt((ra-ra_deg)**2 + (de-dec_deg)**2)
class_star = gaiatest['classprob_dsc_combmod_star'].data
g_sdss = gaiatest['u_jkc_mag'].data
r_sdss = gaiatest['b_jkc_mag'].data
i_sdss = gaiatest['v_jkc_mag'].data
delta_mag = (gaiatest['b_jkc_flux_error'].data / gaiatest['b_jkc_flux'].data)*2.5/np.log(10.)
print(delta_mag)

sel = (class_star > 0.999)  & (np.abs(delta_mag) < 0.004)  # likelihood of being a star
print(len(ra[sel]))

fig, ax = plt.subplots(ncols=2, figsize=(10,5), dpi=120)
ax0=ax[0].scatter(ra[sel], de[sel], s=0.5, c=g_sdss[sel])
ax1=ax[1].scatter(g_sdss[sel]-r_sdss[sel], r_sdss[sel] - i_sdss[sel], s=0.5, c=r_centre[sel])

fig.colorbar(ax0, ax=ax[0], label='g SDSS')
fig.colorbar(ax1, ax=ax[1], label='r')

ax[0].set_xlabel('RA[deg]')
ax[0].set_ylabel('Dec[deg]')
ax[1].set_xlabel('g-r SDSS')
ax[1].set_ylabel('r-i SDSS')
ax[0].invert_xaxis()
#ax[1].invert_yaxis()
ax[0].set_aspect('equal', 'box')
ax[1].set_aspect('equal', 'box')

plt.tight_layout()
plt.show()
